In [17]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS
from mhcgnomes import parse


In [18]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber/"

In [19]:
total_df =  pd.read_csv("{}schreiber_total_data.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))

per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_mt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    per_allele_wt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None

    
ALLELES

['H-2Ld', 'H-2IEd']

# 1. Read all the predictions

In [20]:
import os


In [21]:
per_method_columns = {
    'NetMHC': ['netMHC-3.4_score', 'netMHC-3.4_affinity', 'netMHC-4.0_score',  'netMHC-4.0_affinity','netMHC-4.0_percentile_rank', \
               'netMHCpan-4.1_score_ba', 'netMHCpan-4.1_score_el', 'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 'netMHCpan-4.1_percentile_rank_el',
               'netMHCIIpan-4.0_score_ba', 'netMHCIIpan-4.0_score_el', 'netMHCIIpan-4.0_affinity', 'netMHCIIpan-4.0_percentile_rank_ba', 'netMHCIIpan-4.0_percentile_rank_el',
                ],
    'netMHC-3.4': [ 'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':[ 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':[ 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0' :['score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':[ 'score', 'affinity', 'percentile_rank'],
    'acme':['score'],
    'puffin':['score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood', 'percentile_rank'],
    'mhcseqnet':['score'],
    'mhcattnnet':[ 'score']
}



In [22]:
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber/"
methods =  next(os.walk(PREDICTION_PATH))[1] # read all the folders in a prediction path
methods.remove('NetMHC')
methods.insert(0, 'NetMHC')

#['NetMHC', 'puffin', 'mhcnuggets', 'acme', 'mhcseqnet', 'mhcattnnet']

per_allele_per_method_predictions = dict()

def less_is_good(column):
    if column.find('affinity') >= 0 :
        return True
    if column.find('percentile_rank') >=0:
        return True
    if column in ['epistemic_var', 'aleatoric_var']:
        return True
    return False
        
less_is_good_columns = {'affinity','percentile_rank', 'netMHC-3.4_affinity', 'netMHC-4.0_affinity', 
                                    'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 
                                    'netMHCpan-4.1_percentile_rank_el', 'netMHC-4.0_percentile_rank',  'epistemic_var', 'aleatoric_var' }


for allele_name in ALLELES:
    print(f"{allele_name}:", end=" ")
    per_allele_per_method_predictions[allele_name] = dict()
    for method in methods:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        prediction_df = pd.read_csv(f"{PREDICTION_PATH}{method}/{method}_predictions_allele_{target_allele_name}.csv", sep=';')

        #add diff 

        print(f"{method}", end=" ")
        for score_column in per_method_columns[method]:
            result_score_column = f"{method}_{score_column}" if not (method == 'NetMHC') else score_column
            wt_scores = prediction_df[result_score_column + '_wt'].values
            mt_scores = prediction_df[result_score_column + '_mt'].values
            wt_peptide =  prediction_df['peptide_wt'].values
            # for these columns mt shoud be less than wt. In this case wt/mt will be > 1 and this is good. 
            if less_is_good(score_column):
                advantage = np.array([wt_sc/(mt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            # for these columns mt shoud be greater than wt. In this case mt/wt will be > 1 and this is good. 
            else:
                advantage = np.array([mt_sc/(wt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            prediction_df[f"{result_score_column}_advantage"] = advantage



        per_allele_per_method_predictions[allele_name][method] = prediction_df
    print()
            

H-2Ld: NetMHC mhcnuggets puffin 
H-2IEd: NetMHC mhcnuggets puffin 


# 2. Merge predictions

In [23]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    list_for_allele = list(per_allele_per_method_predictions[allele_name].values())

    result_df = list_for_allele[0].copy()
    for df_to_merge in list_for_allele[1:]:
        result_df = result_df.merge(df_to_merge, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'], how='left')
    result_df = result_df[['allele'] + [column for column in result_df.columns if column.endswith('mt')] 
                          + [column for column in result_df.columns if column.endswith('_advantage')] 
                          + [column for column in result_df.columns if column.endswith('wt')]] 
    per_allele_predictions[allele_name] = result_df


In [24]:
all_methods = list(set([item.split('_')[0] for item in result_df.columns if  item.split('_')[0] not in ['allele', 'peptide']]))
all_methods

['puffin',
 'netMHC-4.0',
 'netMHC-3.4',
 'netMHCpan-4.1',
 'mhcnuggets',
 'netMHCIIpan-4.0']

In [25]:
from config import PER_ORGANISM_PER_CLASS_ALLELES

unsupported_alleles = {method : set() for method in all_methods}

def add_unsupported_allele(method, alleles):
    methods_set = set(all_methods)
    if (method in methods_set):
        unsupported_alleles[method].update(alleles)

for allele_name in ALLELES:
    if defineClass(allele_name) == 'II':
        add_unsupported_allele('netMHCpan-4.1', [allele_name])
        add_unsupported_allele('netMHC-3.4', [allele_name])
        add_unsupported_allele('netMHC-4.0', [allele_name])
        add_unsupported_allele('acme', [allele_name])
    else:
        add_unsupported_allele('netMHCIIpan-4.0', [allele_name])
        
    if not defineOrganism(allele_name) == 'Human':
        add_unsupported_allele('mhcseqnet', [allele_name])
        add_unsupported_allele('mhcattnnet', [allele_name])
        add_unsupported_allele('acme', [allele_name])

#special cases
add_unsupported_allele('acme',["HLA-C*04:01", "HLA-C*03:04"])
add_unsupported_allele('puffin',["HLA-C*04:01"])



def drop_unsupported_columns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        for method in all_methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)]
                df = df.drop(columns=columns_to_drop, errors='ignore')    
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df


def reorder_columns(per_allele_df):
    columns_netmhc34 = [column.strip() for column in df.columns if column.lower.startswith('netmhc-3.4')]
    columns_netmhc40 = [column.strip() for column in df.columns if column.lower.startswith('netmhc-3.4')]
    columns_netmhc34 = [column.strip() for column in df.columns if column.lower.startswith('netmhc-3.4')]


    for allele_name, df in per_allele_df.items():
        for method in all_methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)]
                df = df.drop(columns=columns_to_drop, errors='ignore')    
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df
per_allele_predictions = drop_unsupported_columns(per_allele_predictions)

In [26]:
all_methods

['puffin',
 'netMHC-4.0',
 'netMHC-3.4',
 'netMHCpan-4.1',
 'mhcnuggets',
 'netMHCIIpan-4.0']

## Save total pred to file

In [27]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)